In [0]:
# Demo Setup: Git Folders and Declarative Automation Bundles
# Creates a simple dashboard that students will practice version-controlling.

import re
import json
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import date, timedelta
import random
import time

# --- Schema setup ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "workspace"
schema = f"demo_version_control_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

In [0]:
# --- gold_sales: 500 rows ---
# A small Gold table so the dashboard has live data.

random.seed(33)
spark.sql("DROP TABLE IF EXISTS gold_sales")

regions = ['Northeast', 'Southeast', 'Midwest', 'West']
categories = ['Electronics', 'Clothing', 'Home', 'Sports']

schema_def = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("region", StringType(), False),
    StructField("product_category", StringType(), False),
    StructField("net_revenue", DoubleType(), False)
])

rows = []
for i in range(1, 501):
    order_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
    net_rev = round(random.uniform(20, 500) * random.randint(1, 4), 2)
    rows.append((i, order_date, random.choice(regions), random.choice(categories), net_rev))

df = spark.createDataFrame(rows, schema=schema_def)
df.write.saveAsTable("gold_sales")

In [0]:
# --- Create a dashboard that students will version-control ---
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
parent_path = f"/Users/{username}"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

table_ref = f"`{catalog}`.`{schema}`.`gold_sales`"

dashboard_def = {
    "pages": [{
        "name": "page_overview",
        "displayName": "Sales Overview",
        "layout": [
            {
                "widget": {"name": "w_title", "textbox_spec": "# Regional Sales Report\n\nThis dashboard is used to practice version control workflows."},
                "position": {"x": 0, "y": 0, "width": 6, "height": 1}
            },
            {
                "widget": {
                    "name": "w_bar",
                    "queries": [{"name": "main_query", "query": {"datasetName": "ds_region", "fields": [{"name": "region", "expression": "`region`"}, {"name": "total_revenue", "expression": "`total_revenue`"}], "disaggregated": True}}],
                    "spec": {"version": 3, "widgetType": "bar", "encodings": {"x": {"fieldName": "region", "scale": {"type": "categorical"}, "displayName": "Region"}, "y": {"fieldName": "total_revenue", "scale": {"type": "quantitative"}, "displayName": "Revenue"}}}
                },
                "position": {"x": 0, "y": 3, "width": 6, "height": 3}
            }
        ]
    }],
    "datasets": [
        {"name": "ds_kpi", "displayName": "KPI Summary", "query": f"SELECT ROUND(SUM(net_revenue), 0) AS total_revenue, COUNT(order_id) AS total_orders FROM {table_ref}"},
        {"name": "ds_region", "displayName": "Revenue by Region", "query": f"SELECT region, ROUND(SUM(net_revenue), 2) AS total_revenue FROM {table_ref} GROUP BY region ORDER BY total_revenue DESC"}
    ]
}

dashboard_name = f"Regional Sales Report - Version Control Demo ({clean_username}) {int(time.time())}"
resp = requests.post(
    f"https://{workspace_url}/api/2.0/lakeview/dashboards",
    headers=headers,
    json={"display_name": dashboard_name, "serialized_dashboard": json.dumps(dashboard_def), "parent_path": parent_path}
)

if resp.status_code == 200:
    dashboard_id = resp.json()["dashboard_id"]
    dashboard_path = resp.json().get("path", f"{parent_path}/{dashboard_name}.lvdash.json")
    print(f"\n\u2705 Dashboard created: {dashboard_name}")
    print(f"   ID: {dashboard_id}")
    print(f"   Path: {dashboard_path}")
else:
    print(f"\u26a0\ufe0f  Create returned {resp.status_code}: {resp.text[:200]}")
    dashboard_id = "<CHECK_WORKSPACE>"
    dashboard_path = "<CHECK_WORKSPACE>"

In [0]:
# --- Summary ---
print("SETUP COMPLETE")
print("")
print(f"Catalog:  {catalog}")
print(f"Schema:   {schema}")
print(f"")
print(f"Table: gold_sales - 500 rows")
print(f"")
print(f"Dashboard created (DRAFT):")
print(f"  Name: {dashboard_name}")
print(f"  ID:   {dashboard_id}")
print(f"  Path: {dashboard_path}")